# DA-TDMPC2: Nominal Evaluation Sweep (Task T007)
**Owner:** M1 · **Tier:** T1 (Google Colab GPU)

Evaluates `mt30-19M` and `mt30-5M` on all 30 MT30 tasks under nominal conditions (`--shift none`).

In [ ]:
# 1. Parameters (edit per sweep)
COMMIT = 'dev_shahriar'   # Branch or commit to evaluate
SHARD = '1/1'             # '1/1' for full run, or '1/3' if sharding
PHASE = 'P0_nominal'
DEVICE = 'cuda'           # Use GPU acceleration
DRIVE = '/content/drive/MyDrive/DA-TDMPC2'
REPO_URL = 'https://github.com/Shahrcasm/tdmpc2-shift-adaptation.git'


In [ ]:
# 2. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# 3. Clone Repository
import getpass, subprocess
token = getpass.getpass('GitHub token (read access; not stored): ')
auth_url = REPO_URL.replace('https://', 'https://' + token + '@')
subprocess.run(['git', 'clone', '-q', '-b', COMMIT, auth_url, '/content/tdmpc2-shift-adaptation'], check=True)
print('Repository cloned successfully!')
del token, auth_url


In [ ]:
# 4. Install Dependencies
%cd /content/tdmpc2-shift-adaptation
!pip install -q torch==2.7.1
!pip install -q -r environment/requirements-colab.txt
!pip install -q -e . --no-deps
!python -c 'import torch; print(f"PyTorch {torch.__version__}, CUDA: {torch.cuda.is_available()}")'


In [ ]:
# 5. Checkpoints (from Drive or Hugging Face)
!mkdir -p checkpoints
!cp {DRIVE}/checkpoints/mt30-*.pt checkpoints/ 2>/dev/null || python scripts/get_checkpoints.py --models mt30-5M mt30-19M
!python scripts/get_checkpoints.py --models mt30-5M mt30-19M --verify-only


In [ ]:
# 6. Fast Smoke Test
!python scripts/smoke_test.py --mode tiny


In [ ]:
# 7. Run Primary Checkpoint (mt30-19M)
!python scripts/eval.py --ckpt checkpoints/mt30-19M.pt --task all --shift none --episodes 10 --seeds 0 1 2 --out {DRIVE}/results/{PHASE}/ --resume --shard {SHARD} --tier T1 --device {DEVICE}


In [ ]:
# 8. Run Secondary Checkpoint (mt30-5M)
!python scripts/eval.py --ckpt checkpoints/mt30-5M.pt --task all --shift none --episodes 10 --seeds 0 1 2 --out {DRIVE}/results/{PHASE}/ --resume --shard {SHARD} --tier T1 --device {DEVICE}
